# Tasks

- Implement a transformer end-to-end
- Implement causal, cross, and self attention
- Implement flash attention
- Implement the attention backward pass
- Implement an MLP forward and backward pass
   - Mixture of Experts
- Implement a simple training loop with SGD in PyTorch or JAX

In [ ]:
import torch
from torch import Tensor
from torch.nn import Module
from dataclasses import dataclass
import torch.nn.functional as F

In [4]:
@dataclass
class model:
  D: int # embedding dim
  H: int # hidden dim
  N: int # num Q heads
  K: int # num KV heads
  F: int # up-projection
  L: int # num layers

In [5]:
def attention(x_q, x_k, x_v):
    # x_q: [B, T, N, H]
    # x_k: [B, S, K, H]
    # x_v: [B, S, K, H]

    B, T, N, H = x_q.shape
    _, S, K, _ = x_k.shape

    assert N % K == 0 # GQA requirement
    g = N // K
    q = x_q.permute(0, 2, 1, 3).reshape(B, g, K, T, H)
    k = x_k.permute(0, 2, 3, 1).unsqueeze(1)  # [B, 1, K, H, S]
    v = x_v.permute(0, 2, 1, 3).unsqueeze(1)  # [B, 1, K, S, H]

    scores = torch.matmul(q, k) * (1.0 / torch.sqrt(H))
    probs = torch.softmax(scores, dim=-1)

    out = torch.matmul(probs, v)  # [B, g, K, T, H]
    out = out.permute(0, 3, 1, 2, 4).reshape(B, T, N, H)

    return out

In [ ]:
def dense_transformer(
    x: Tensor,
    Wq: Tensor,     # [D, N, H]
    Wk: Tensor,     # [D, K, H]
    Wv: Tensor,     # [D, K, H]
    Wo: Tensor,     # [N, H, D]
    Wup: Tensor,    # [D, F]
    Wdown: Tensor,  # [F, D]
) -> Tensor:
  # x: [B, T, D]
  B, T, D = x.shape
  N, H = Wq.shape[-2:]
  K, _ = Wk.shape[-2:]
  F = Wup.shape[-1]

  ## QKV Projections
  Wq_flat = Wq.reshape(D, N*H)
  Wk_flat = Wk.reshape(D, K*H)
  Wv_flat = Wv.reshape(D, K*H)
  x_q = F.linear(x, Wq_flat.T) # [B, T, N*H]
  x_q = x_q.reshape(B, T, N, H) # [B, T, N, H]
  x_k = F.linear(x, Wk_flat.T) # [B, T, K*H]
  x_k = x_k.reshape(B, T, K, H) # [B, T, K, H]
  x_v = F.linear(x, Wv_flat.T) # [B, T, K*H]
  x_v = x_v.reshape(B, T, K, H) # [B, T, K, H]

  ## Attention
  output = attention(x_q, x_k, x_v) # [B, T, N, H]

  ## Output Projection
  output = F.linear(output.reshape(B, T, N*H), Wo.T)

  ## Add and Normalize
  output = output + x
  output = F.layer_norm(output, output.shape[-1:])

  ## MLP -> [B, T, D]
  up_proj = F.linear(output, Wup.T) # [B, T, F]
  up_proj = F.gelu(up_proj)
  down_proj = F.linear(up_proj, Wdown.T) # [B, T, D]

  ## Add + Norm
  output = down_proj + output
  output = F.layer_norm(output, D)

  return output # [B, T, D]


In [ ]:
## More Modern PyTorch (unflatten, movedim, einsum)

import math
import torch
import torch.nn.functional as F
import torch.nn as nn
from torch import Tensor


def attention_gqa(x_q: Tensor, x_k: Tensor, x_v: Tensor) -> Tensor:
    # x_q: [B, T, N, H]
    # x_k: [B, S, K, H]
    # x_v: [B, S, K, H]

    B, T, N, H = x_q.shape
    _, S, K, _ = x_k.shape

    assert x_k.shape == (B, S, K, H)
    assert x_v.shape == (B, S, K, H)
    assert N % K == 0

    G = N // K

    # [B, T, N, H] -> [B, G, K, T, H]
    q = x_q.unflatten(dim=2, sizes=(G, K)).movedim(1, 3)

    # scores: [B, G, K, T, S]
    scores = torch.einsum("bgkth,bskh->bgkts", q, x_k)
    scores = scores * (1.0 / math.sqrt(H))

    probs = scores.softmax(dim=-1)

    # out: [B, G, K, T, H]
    out = torch.einsum("bgkts,bskh->bgkth", probs, x_v)

    # [B, G, K, T, H] -> [B, T, N, H]
    return out.movedim(3, 1).flatten(2, 3)


def dense_transformer(
    x: Tensor,
    Wq: Tensor,     # [D, N, H]
    Wk: Tensor,     # [D, K, H]
    Wv: Tensor,     # [D, K, H]
    Wo: Tensor,     # [N, H, D]
    Wup: Tensor,    # [D, M]
    Wdown: Tensor,  # [M, D]
) -> Tensor:
    # x: [B, T, D]

    B, T, D = x.shape
    _, N, H = Wq.shape
    _, K, _ = Wk.shape
    M = Wup.shape[-1]

    assert Wq.shape == (D, N, H)
    assert Wk.shape == (D, K, H)
    assert Wv.shape == (D, K, H)
    assert Wo.shape == (N, H, D)
    assert Wup.shape == (D, M)
    assert Wdown.shape == (M, D)

    # QKV projections
    x_q = torch.einsum("btd,dnh->btnh", x, Wq)
    x_k = torch.einsum("btd,dkh->btkh", x, Wk)
    x_v = torch.einsum("btd,dkh->btkh", x, Wv)

    # Attention: [B, T, N, H]
    attn = attention_gqa(x_q, x_k, x_v)

    # Output projection: [B, T, D]
    output = torch.einsum("btnh,nhd->btd", attn, Wo)

    # Residual + norm
    output = F.layer_norm(output + x, normalized_shape=(D,))

    # MLP
    hidden = torch.einsum("btd,dm->btm", output, Wup)
    hidden = F.gelu(hidden)
    mlp = torch.einsum("btm,md->btd", hidden, Wdown)

    # Residual + norm
    output = F.layer_norm(output + mlp, normalized_shape=(D,))

    return output

In [ ]:
# REUSABLE, TRAINABLE, EXTENSIBLE, PORTABLE PyTorch
# nn.Module
"""
- parameters are registered automatically
- model.to(device, dtype) moves everything correctly
- model.parameters() works with optimizers
- state_dict() / load_state_dict() gives serialization
- train() / eval() mode propagates through submodules
- torch.compile(model) has a clean callable object to wrap
- blocks can be stacked, named, inspected, frozen, shared, or checkpointed
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor


class GQAAttention(nn.Module):
    def forward(self, q: Tensor, k: Tensor, v: Tensor) -> Tensor:
        # q: [B, T, N, H]
        # k: [B, S, K, H]
        # v: [B, S, K, H]

        B, T, N, H = q.shape
        _, S, K, _ = k.shape

        assert k.shape == (B, S, K, H)
        assert v.shape == (B, S, K, H)
        assert N % K == 0

        G = N // K

        # [B, T, N, H] -> [B, G, K, T, H]
        q = q.unflatten(dim=2, sizes=(G, K)).movedim(1, 3)

        # [B, G, K, T, S]
        scores = torch.einsum("bgkth,bskh->bgkts", q, k)
        scores = scores * (1.0 / math.sqrt(H))

        probs = scores.softmax(dim=-1)

        # [B, G, K, T, H]
        out = torch.einsum("bgkts,bskh->bgkth", probs, v)

        # [B, G, K, T, H] -> [B, T, N, H]
        return out.movedim(3, 1).flatten(2, 3)


class DenseTransformerBlock(nn.Module):
    def __init__(
        self,
        d_model: int,
        n_heads: int,
        head_dim: int,
        n_kv_heads: int,
        ffn_dim: int,
        *,
        affine_norm: bool = False,
    ):
        super().__init__()

        assert n_heads % n_kv_heads == 0

        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = head_dim
        self.n_kv_heads = n_kv_heads
        self.ffn_dim = ffn_dim

        self.Wq = nn.Parameter(torch.empty(d_model, n_heads, head_dim))
        self.Wk = nn.Parameter(torch.empty(d_model, n_kv_heads, head_dim))
        self.Wv = nn.Parameter(torch.empty(d_model, n_kv_heads, head_dim))
        self.Wo = nn.Parameter(torch.empty(n_heads, head_dim, d_model))

        self.Wup = nn.Parameter(torch.empty(d_model, ffn_dim))
        self.Wdown = nn.Parameter(torch.empty(ffn_dim, d_model))

        # note we are saving the Attention Sub-module as a class member
        self.attn = GQAAttention()

        # Use elementwise_affine=False to match F.layer_norm with no weight/bias.
        self.attn_norm = nn.LayerNorm(d_model, elementwise_affine=affine_norm)
        self.mlp_norm = nn.LayerNorm(d_model, elementwise_affine=affine_norm)

        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.xavier_uniform_(self.Wq.flatten(1))
        nn.init.xavier_uniform_(self.Wk.flatten(1))
        nn.init.xavier_uniform_(self.Wv.flatten(1))
        nn.init.xavier_uniform_(self.Wo.flatten(0, 1))
        nn.init.xavier_uniform_(self.Wup)
        nn.init.xavier_uniform_(self.Wdown)

    def forward(self, x: Tensor) -> Tensor:
        # x: [B, T, D]

        q = torch.einsum("btd,dnh->btnh", x, self.Wq)
        k = torch.einsum("btd,dkh->btkh", x, self.Wk)
        v = torch.einsum("btd,dkh->btkh", x, self.Wv)

        attn = self.attn(q, k, v)
        attn = torch.einsum("btnh,nhd->btd", attn, self.Wo)

        x = self.attn_norm(x + attn)

        hidden = torch.einsum("btd,dm->btm", x, self.Wup)
        hidden = F.gelu(hidden)
        mlp = torch.einsum("btm,md->btd", hidden, self.Wdown)

        x = self.mlp_norm(x + mlp)

        return x

In [ ]:
### Usage

block = DenseTransformerBlock(
    d_model=512,
    n_heads=8,
    head_dim=64,
    n_kv_heads=2,
    ffn_dim=2048,
)

x = torch.randn(4, 128, 512)
y = block(x)

optimizer = torch.optim.AdamW(block.parameters(), lr=3e-4)

block = block.to("cuda", dtype=torch.bfloat16)

torch.save(block.state_dict(), "block.pt")

block2 = DenseTransformerBlock(512, 8, 64, 2, 2048)
block2.load_state_dict(torch.load("block.pt"))

In [ ]:
class SomeModel(nn.Module):
    def __init__(
      self,
      num_layers: int,
      d_model: int,
      n_heads: int,
      head_dim: int,
      n_kv_heads: int,
      ffn_dim: int,
      *,
      affine_norm: bool = False,
      vocab_size: int,
    ):
        super().__init__()
        self.layers = nn.ModuleList(
            [
                DenseTransformerBlock(
                    d_model,
                    n_heads,
                    head_dim,
                    n_kv_heads,
                    ffn_dim,
                    affine_norm=affine_norm,
                )
                for _ in range(num_layers)
            ]
        )
        self.unembed = nn.Linear(d_model, vocab_size)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        for layer in self.layers:
            layer.reset_parameters()
        nn.init.zeros_(self.unembed.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return self.unembed(x)

In [ ]:
class FlashGQA(nn.Module):
    def __init__(self, tileSize):
        super().__init__()
        self.tile_size = tileSize
    
    def forward(self, 
                x_q: Tensor,
                x_k: Tensor,
                x_v: Tensor,
        ) -> Tensor:
        B, T, N, H = x_q.shape
        _, _, K, _ = x_v.shape
        assert x_v.shape[1] == T # Prefill only
        TILES = T // self.tile_size
        g = N // K
        x_q = x_q.reshape(B, TILES, self.tile_size, g, K, H) # [B, TILES, TILE_SIZE, g, K, H]
        x_k = x_k.reshape(B, TILES, self.tile_size, 1, K, H) # [B, TILES, TILE_SIZE, 1, K, H]
        x_v = x_v.reshape(B, TILES, self.tile_size, 1, K, H) # [B, TILES, TILE_SIZE, 1, K, H]
        x_q = x_q.permute(0, 3, 4, 1, 2, 5) # [B, g, K, TILES, TILE_SIZE, H]
        x_k = x_k.permute(0, 3, 4, 1, 2, 5) # [B, 1, K, TILES, TILE_SIZE, H]
        x_v = x_v.permute(0, 3, 4, 1, 2, 5) # [B, 1, K, TILES, TILE_SIZE, H]
        x_out = torch.zeros(B, g, K, TILES, self.tile_size, H)
        # QTiles Outer-loop
        for qtile in range(TILES):
            attn_out = torch.zeros(B, g, K, self.tile_size, H)
            max_old = torch.full((B, g, K, self.tile_size, 1), -float(torch.inf))
            sum_old = torch.zeros(B, g, K, self.tile_size, 1)
            x_q_tile = x_q[:, :, :, qtile, :, :] # [B, g, K, 1, TILE_SIZE, H]
            # KVTiles in Inner-loop
            for kvtile in range(TILES):
                if kvtile > qtile:
                    continue
                x_k_tile = x_k[:, :, :, kvtile, :, :] # [B, 1, K, 1, TILE_SIZE, H]
                x_v_tile = x_v[:, :, :, kvtile, :, :] # [B, 1, K, 1, TILE_SIZE, H]
                scores_tile = torch.matmul(x_q_tile, x_k_tile.transpose(-2, -1)) # [B, g, K, TILE_SIZE, TILE_SIZE]
                scores_tile = scores_tile / math.sqrt(H)
                if kvtile == qtile:
                    causal_mask = torch.triu(torch.ones(self.tile_size, self.tile_size, device=scores_tile.device, dtype=torch.bool),diagonal=1)
                    scores_tile = scores_tile.masked_fill(causal_mask, -float("inf"))
                max_new = scores_tile.max(dim=-1, keepdim=True).values # [B, g, K, TILE_SIZE, 1]
                max_new = torch.maximum(max_old, max_new)
                probs_tile = torch.exp(scores_tile - max_new) # [B, g, K, TILE_SIZE, TILE_SIZE]
                sum_new = probs_tile.sum(dim=-1, keepdim=True) # [B, g, K, TILE_SIZE, 1]
                sum_new = sum_old * torch.exp(max_old - max_new) + sum_new
                out_new = torch.matmul(probs_tile / sum_new, x_v_tile) # [B, g, K, TILE_SIZE, H]
                attn_out = attn_out * (sum_old / sum_new) * torch.exp(max_old - max_new) + out_new # [B, g, K, TILE_SIZE, H]
                max_old = max_new
                sum_old = sum_new
            x_out[:, :, :, qtile, :, :] = attn_out # [B, g, K, TILES, TILE_SIZE, H]
        x_out = x_out.reshape(B, N, T, H)
        x_out = x_out.permute(0, 2, 1, 3)
        return x_out

In [ ]:
class MoEMLP(nn.Module):
    def __init__(
        self,
        d_model: int,
        ffn_dim: int,
        num_experts: int,
        top_k: int,
    ):
        super().__init__()

        assert top_k > 0
        assert top_k <= num_experts

        self.d_model = d_model
        self.ffn_dim = ffn_dim
        self.num_experts = num_experts
        self.top_k = top_k

        # Router maps each token to logits over E experts.
        self.Wrouter = nn.Parameter(torch.empty(d_model, num_experts))

        # Per-expert MLP weights: expert e owns Wup[e] and Wdown[e].
        self.Wup = nn.Parameter(torch.empty(num_experts, d_model, ffn_dim))
        self.Wdown = nn.Parameter(torch.empty(num_experts, ffn_dim, d_model))

        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.xavier_uniform_(self.Wrouter)
        for expert in range(self.num_experts):
            nn.init.xavier_uniform_(self.Wup[expert])
            nn.init.xavier_uniform_(self.Wdown[expert])

    def forward(self, x: Tensor) -> Tensor:
        # x: [B, T, D]
        B, T, D = x.shape
        assert D == self.d_model

        x_flat = x.reshape(B * T, D)  # [BT, D]

        # Router logits and TopK expert selection per token.
        router_logits = torch.matmul(x_flat, self.Wrouter)  # [BT, E]
        topk_logits, topk_experts = torch.topk(router_logits, k=self.top_k, dim=-1)  # [BT, TopK]
        topk_weights = torch.softmax(topk_logits, dim=-1)  # [BT, TopK]

        out_flat = x_flat.new_zeros(B * T, D)

        # Route selected token/expert pairs through each expert.
        # topk_experts[t, r] stores the expert id chosen for flattened token t
        # at router rank r, where r is in [0, TopK).
        for expert in range(self.num_experts):
            # selected[t, r] is True when flattened token t chose this expert
            # as its r-th routed expert. Shape: [BT, TopK].
            selected = topk_experts == expert  # [BT, TopK]

            # token_idx and topk_idx are parallel 1D tensors. For every selected
            # pair i, token_idx[i] says which token to run, and topk_idx[i]
            # says which TopK router weight belongs to this expert choice.
            token_idx, topk_idx = selected.nonzero(as_tuple=True)
            if token_idx.numel() == 0:
                continue

            # Gather only the tokens assigned to this expert, making a compact
            # mini-batch for expert-specific matrix multiplies.
            expert_input = x_flat[token_idx]  # [N_selected, D]

            # Apply this expert's own MLP: D -> F -> D.
            hidden = torch.matmul(expert_input, self.Wup[expert])  # [N_selected, F]
            hidden = F.gelu(hidden)
            expert_out = torch.matmul(hidden, self.Wdown[expert])  # [N_selected, D]

            # Gather each selected token's router probability for this expert.
            # The unsqueeze broadcasts the scalar route weight across D channels.
            weights = topk_weights[token_idx, topk_idx].unsqueeze(-1)  # [N_selected, 1]

            # Add weighted expert outputs back into their original token rows.
            # index_add_ handles accumulation because each token contributes
            # TopK expert outputs to the same out_flat[token_idx] row.
            out_flat.index_add_(0, token_idx, expert_out * weights)

        return out_flat.reshape(B, T, D)
